<a href="https://colab.research.google.com/github/codeunitdev/natlas-devkit/blob/main/notebooks/N_ATLAS_DevKit_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# N-ATLaS DevKit — Colab demo

Run Nigeria's open language model **N-ATLaS** on a free Colab GPU, then use it three ways:
the **Python SDK**, an **OpenAI-compatible API**, and a **web playground**.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Sign in at huggingface.co and accept the terms on the [NCAIR1/N-ATLaS](https://huggingface.co/NCAIR1/N-ATLaS) model page.
3. Create a *read* token at huggingface.co/settings/tokens and add it in Colab's **🔑 Secrets** panel as `HF_TOKEN`.

*N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

## 1. Install

In [11]:
!pip install -q "natlas-devkit[all] @ git+https://github.com/codeunitdev/natlas-devkit.git" openai

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Hugging Face login

In [12]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Load N-ATLaS in 4-bit (about 6 GB of GPU memory; first download takes a few minutes)

In [13]:
from natlas import NAtlas
llm = NAtlas(quantization="4bit")
print("Loaded", llm.model_id)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded NCAIR1/N-ATLaS


## 4. Python SDK — one model, four languages

In [14]:
prompts = {
    "Hausa":   "menene ake nufi da gwagwarmaya",
    "Yoruba":  "Báwo ni mo ṣe lè dáàbò bo ara mi lọ́wọ́ ibà?",
    "Igbo":    "Kedu ka m ga-esi chebe onwe m pụọ n'ọrịa ịba?",
    "English": "Explain in simple terms how a farmer can save money with a cooperative.",
}
for lang, q in prompts.items():
    print(f"=== {lang} ===\nQ: {q}\nA: {llm.ask(q, max_new_tokens=250)}\n")

=== Hausa ===
Q: menene ake nufi da gwagwarmaya
A: Gwagwarmaya na iya haɗawa da juriya, tawaye, ko kuma yaƙin neman canji ga wani buri, manufa, ko dalili. Sau da yawa yana bukatar sadaukarwa, juriya, da kuma shawo kan cikas don cimma burinsu. Gwagwarmayar mutum na iya bambanta daga gwaji na sirri zuwa fafutuka na zamantakewa ko siyasa.

=== Yoruba ===
Q: Báwo ni mo ṣe lè dáàbò bo ara mi lọ́wọ́ ibà?
A: Daidaitawa daidai ga abin da kuke nema. Don kariya daga bala'i, yana da mahimmanci don shirya yadda ya kamata. Ga wasu matakai masu tasiri:

1. **Shirye-shiryen Tsari**: Ka sanar da kanka game da nau'ikan bala'o'i daban-daban (gajimare, ambaliyar ruwa, girgizar kasa, wuta) da ke faruwa a yankinku. Fahimtar haɗarin ku na iya taimaka muku yanke shawara mai kyau.

2. **Kirkira Kula da Bala’i**: Sanya tsarin gargaɗi ko samun manhaja ta wayar hannu da aka yi wa lakabi da gargaɗi na bala'i a yankinki. Wannan zai ba ka damar samun isasshen lokaci don daukar mataki idan wani abu ya faru.

3. **Wu

### Translation and forced reply language

In [15]:
print(llm.translate("Wash your hands with soap before eating.", to="Hausa"))
print(llm.ask("What are the early signs of malaria?", language="Igbo", max_new_tokens=200))

Ku wanke hannayenku da sabulu kafin ku ci abinci.
Ihe mbụ nke ịhịa aka n'ahụ bụ ihe mgbaàmà dị ka ahụ ọkụ na mgbu n'akụkụ ebe anwụnta na-abanye n'ime akpụkpọ ahụ.


### Streaming

In [16]:
for piece in llm.stream_chat([{"role": "user", "content": "Give me three tips for saving money as a student."}], max_new_tokens=250):
    print(piece, end="", flush=True)

Sure! Here are three tips for saving money as a student:

1. **Budgeting:** Create and follow a budget that outlines your monthly income and expenses. This will help you identify areas where you can cut back on unnecessary spending.

2. **Cook Your Own Meals:** Instead of buying lunch at school cafeterias or restaurants, consider cooking your own meals. It may seem easier to grab something quickly, but it can be expensive over time. Cooking your own meals allows you to save money and have more control over what goes into your food.

3. **Avoid Impulse Purchases:** Be mindful of impulse buys, especially when shopping online. Take some time to think about whether you really need an item before adding it to your cart. You'll likely find that you don't need many things you initially thought were necessary!

By following these tips, you can save money effectively while still enjoying your college experience.

## 5. OpenAI-compatible API

Start the API server in the background (reusing the loaded model), then call it with the **official OpenAI client**.
Any app built for OpenAI switches to N-ATLaS by changing `base_url`.

In [17]:
import threading, time, uvicorn
from natlas.server import create_app

server = uvicorn.Server(uvicorn.Config(create_app(engine=llm), host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local")
resp = client.chat.completions.create(
    model="n-atlas",
    messages=[{"role": "user", "content": "menene ake nufi da gwagwarmaya"}],
    max_tokens=200,
)
print(resp.choices[0].message.content)
print(resp.usage)

ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


Gwagwarmaya na iya nufin abubuwa daban-daban dangane da yanayinta. A al'ada, ana iya fahimtar ta a matsayin kokari, juriya, ko kuma yaki don cimma wata manufa, samun nasara, ko shawo kan kalubale. Ana amfani da wannan ra’ayi a fannonin rayuwa iri-iri, kamar wasanni, kasuwanci, ko ci gaban mutum.
CompletionUsage(completion_tokens=118, prompt_tokens=95, total_tokens=213, completion_tokens_details=None, prompt_tokens_details=None)


## 6. Web playground (opens a temporary public link)

In [18]:
from natlas.playground import build_demo
build_demo(engine=llm).queue().launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://662da680b07fd7fafd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
